In [ ]:
# Cell 1 — Instalación de dependencias e imports
import subprocess, sys
for pkg in ['requests', 'openpyxl', 'pdfplumber', 'playwright', 'xlwings', 'nest_asyncio', 'python-dotenv']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=False)

import os, re, math, shutil, logging, json, getpass
from pathlib import Path
from datetime import datetime, timedelta
import requests
import openpyxl
import pdfplumber
import xlwings as xw
import nest_asyncio
import asyncio
import unicodedata
from playwright.async_api import async_playwright
from dotenv import load_dotenv

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

nest_asyncio.apply()  # Permite asyncio.run() dentro del event loop de Jupyter

PROJECT_ROOT = Path(r'c:\dev\Asistencia')
load_dotenv(PROJECT_ROOT / '.env')

print('Imports OK')

In [ ]:
# Cell 2 — Obtener usuario y configurar log

def obtener_usuario() -> str:
    return os.environ.get('USERNAME') or getpass.getuser()

def setup_log(username: str) -> None:
    log_dir = PROJECT_ROOT / 'logs'
    log_dir.mkdir(parents=True, exist_ok=True)
    log_file = log_dir / f'{datetime.now().strftime("%Y-%m-%d")}.log'

    root = logging.getLogger()
    root.setLevel(logging.INFO)
    root.handlers.clear()
    fmt = logging.Formatter('%(asctime)s.%(msecs)03d %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
    fh = logging.FileHandler(log_file, encoding='utf-8')
    fh.setFormatter(fmt)
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    root.addHandler(fh)
    root.addHandler(sh)

USERNAME = obtener_usuario()
setup_log(USERNAME)
logging.info(f'Bot Asistencia - Inicio Ejecucion - Usuario: {USERNAME}')

In [ ]:
# Cell 3 — Funciones helper (definidas una sola vez, usadas en todo el notebook)
#
# Correcciones aplicadas respecto al JS original:
#   Fix #1  : format_fecha else retorna '' (JS retornaba new Date() objeto)
#   Fix #5  : safe_len usa str(val) antes de len() (JS .length sobre numero = undefined)
#   Fix #6  : fec_puntos usa re.sub global (JS .replace solo primera ocurrencia)
#   Fix #8/9: slices explícitos h[2:4] y date[8:12] (JS usaba quirk substring con índices invertidos)

def format_hora(h: str) -> str:
    """'HHMM' -> 'HH:MM'. Fix #9: h[2:4] explícito en lugar del quirk JS h.substring(4,2)."""
    s = str(h)
    if len(s) >= 4:
        return s[0:2] + ':' + s[2:4]
    return ''

def format_fecha(f: str) -> str:
    """'YYYYMMDD...' -> 'DD-MM-YYYY'. Fix #1: else retorna '' no objeto datetime."""
    s = str(f)
    if len(s) >= 8:
        return s[6:8] + '-' + s[4:6] + '-' + s[0:4]
    return ''

def format_rut(ru: str) -> str:
    """Formatea RUT chileno a XX.XXX.XXX-X."""
    new_rut = str(ru).replace('.', '').replace('-', '').strip().lower()
    if not new_rut:
        return ''
    last_digit = new_rut[-1]
    rut_digits = new_rut[:-1]
    fmt = ''
    for i, c in enumerate(reversed(rut_digits)):
        fmt = c + fmt
        if (i + 1) % 3 == 0:
            fmt = '.' + fmt
    return fmt + '-' + last_digit

def reemplazar(rut: str) -> str:
    """Quita puntos, mantiene guión. '12.345.678-9' -> '12345678-9'."""
    return str(rut).replace('.', '').strip()

def rut_numerico(rut: str) -> str:
    """Quita toda la puntuación. '26.662.899-4' -> '266628994'."""
    return str(rut).replace('.', '').replace('-', '').strip()

def safe_len(val) -> int:
    """Fix #5: si val es número, JS .length devuelve undefined. Python convierte a str primero."""
    if val is None:
        return 0
    return len(str(val))

def normalize_str(s: str) -> str:
    """Quita tildes/diacríticos y convierte a minúsculas para comparaciones."""
    return unicodedata.normalize('NFD', str(s)).encode('ascii', 'ignore').decode('ascii').lower()

def extraer_hora_punch(date_str: str) -> str:
    """Fix #8: date_str es 'YYYYMMDDHHMMSS' (14 chars). Extrae HHMM[8:12] explícitamente.
    JS usaba .substring(8).substring(4,0) que funciona solo por el quirk de indices invertidos."""
    s = str(date_str)
    if len(s) >= 12:
        return format_hora(s[8:12])
    return ''

# Validaciones rápidas
assert format_hora('0730') == '07:30', 'format_hora KO'
assert format_hora('') == '', 'format_hora vacío KO'
assert format_fecha('20250822') == '22-08-2025', 'format_fecha KO'
assert format_fecha('') == '', 'format_fecha vacío KO (Fix #1)'
assert format_rut('26662899-4') == '26.662.899-4', 'format_rut KO'
assert rut_numerico('26.662.899-4') == '266628994', 'rut_numerico KO'
assert safe_len(0) == 1, 'safe_len número KO (Fix #5)'
assert safe_len('') == 0, 'safe_len vacío KO'
assert extraer_hora_punch('20250822083045') == '08:30', 'extraer_hora_punch KO (Fix #8)'

print('Helpers OK — todos los asserts pasaron')

In [ ]:
# Cell 4 — Respaldar carpeta Downloads (archivos y carpetas)

def respaldar_downloads(username: str) -> None:
    downloads = Path(f'C:/Users/{username}/Downloads')
    respaldo = downloads / 'respaldo descargas'
    respaldo.mkdir(parents=True, exist_ok=True)

    copiados_archivos = []
    copiados_carpetas = []

    for item in downloads.iterdir():
        if item == respaldo:
            continue  # No respaldar la carpeta de respaldo en sí misma
        if item.is_file():
            try:
                shutil.copy2(item, respaldo / item.name)
                logging.info(f'MoverArchivosDescarga - Copiar archivo a respaldo: {item.name}')
                copiados_archivos.append(item)
            except Exception as e:
                logging.error(f'MoverArchivosDescarga - Error copiando archivo {item.name}: {e}')
        elif item.is_dir():
            dest_dir = respaldo / item.name
            try:
                if dest_dir.exists():
                    shutil.rmtree(dest_dir)
                shutil.copytree(item, dest_dir)
                logging.info(f'MoverArchivosDescarga - Copiar carpeta a respaldo: {item.name}')
                copiados_carpetas.append(item)
            except Exception as e:
                logging.error(f'MoverArchivosDescarga - Error copiando carpeta {item.name}: {e}')

    for f in copiados_archivos:
        try:
            f.unlink()
        except Exception as e:
            logging.error(f'MoverArchivosDescarga - Error eliminando archivo {f.name}: {e}')

    for d in copiados_carpetas:
        try:
            shutil.rmtree(d)
        except Exception as e:
            logging.error(f'MoverArchivosDescarga - Error eliminando carpeta {d.name}: {e}')

    logging.info(f'MoverArchivosDescarga - Archivos respaldados: {len(copiados_archivos)}, Carpetas: {len(copiados_carpetas)}')

respaldar_downloads(USERNAME)

In [ ]:
# Cell 5 — ObtenerToken → TOKEN

async def obtener_token() -> str:
    url = 'https://customerapi.geovictoria.com/api/v1/Login'
    try:
        resp = requests.post(
            url,
            json={
                'User':     os.environ['GEO_USER'],
                'Password': os.environ['GEO_PASSWORD'],
            },
            headers={'Accept': 'application/json', 'Content-Type': 'application/json'},
            verify=False,
            timeout=30
        )
        resp.raise_for_status()
        token = resp.json().get('token', '')
        logging.info(f'LoginGeoVictoriaWS - Obtener token: {token}')
        return token
    except Exception as e:
        logging.error(f'LoginGeoVictoriaWS - Error en servicio Login: {e}')
        return ''

TOKEN = asyncio.run(obtener_token())
assert len(TOKEN) > 10, f'Token vacío o inválido — verificar credenciales GeoVictoria'

In [ ]:
# Cell 6 — ObtenerUsuarios + GenerarCarpetas
#
# Fix #2 : variable única 'lista' (JS alternaba 'listUsuarios'/'listaUsuarios')
# Fix #10: variable 'req' muerta eliminada

OAUTH_HEADER = (
    f'OAuth oauth_consumer_key="{os.environ["GEO_OAUTH_CONSUMER_KEY"]}",'
    f'oauth_signature_method="HMAC-SHA1",'
    f'oauth_timestamp="{os.environ["GEO_OAUTH_TIMESTAMP"]}",'
    f'oauth_nonce="{os.environ["GEO_OAUTH_NONCE"]}",'
    f'oauth_version="1.0",'
    f'oauth_signature="{os.environ["GEO_OAUTH_SIGNATURE"]}"'
)

async def obtener_usuarios(token: str) -> list:
    url = 'https://apiv3.geovictoria.com/api/User/List'
    try:
        resp = requests.post(
            url,
            headers={
                'Accept': 'application/json',
                'Content-Type': 'application/json',
                'Authorization': OAUTH_HEADER
            },
            timeout=30
        )
        resp.raise_for_status()
        data = resp.json()
        logging.info(f'ObtenerUsuarios - Usuarios obtenidos: {len(data)}')
        lista = [
            item['Identifier']
            for item in data
            if item.get('Enabled') == 1
            and 'casino' not in (item.get('GroupDescription') or '').lower()
        ]
        logging.info(f'ObtenerUsuarios - Usuarios validos: {len(lista)}')
        return lista
    except Exception as e:
        logging.error(f'ObtenerUsuarios - Error en servicio Usuarios: {e}')
        return []

def generar_carpetas(username: str) -> dict:
    lic = PROJECT_ROOT / 'Licencias'
    carpetas = {
        'downloads':             Path(f'C:/Users/{username}/Downloads'),
        'licencias':             lic,
        'formato_permisos':      lic / 'Formato_permisos',
        'formato_final':         lic / 'Formato_final_licencias_medicas',
        'sin_tramitar_imed':     lic / 'Licencias_sin_tramitar/imed',
        'sin_tramitar_medipass': lic / 'Licencias_sin_tramitar/medipass',
        'tramitadas_imed':       lic / 'Licencias_tramitadas/imed',
        'tramitadas_medipass':   lic / 'Licencias_tramitadas/medipass',
        'respaldo':              Path(f'C:/Users/{username}/Downloads/respaldo descargas'),
    }
    for p in carpetas.values():
        p.mkdir(parents=True, exist_ok=True)
    logging.info(f'GenerarCarpetas - Carpetas creadas/verificadas: {len(carpetas)}')
    return carpetas

LISTA_USUARIOS = asyncio.run(obtener_usuarios(TOKEN))
CARPETAS = generar_carpetas(USERNAME)
assert len(LISTA_USUARIOS) > 0, f'Lista de usuarios vacía — verificar API GeoVictoria'

In [ ]:
# Cell 7 — Paginar + calcular ventana de fechas
#
# Fix #7: math.ceil explícito (JS usaba división fraccionaria que funcionaba por accidente)
# Fix #3: paginar() retorna la lista directamente (JS tenía 'return listaPaginada' siendo variable local 'l')

def paginar(array: list, page_size: int = 100) -> list:
    """Fix #7: math.ceil garantiza que el último bloque parcial se incluya."""
    n = math.ceil(len(array) / page_size) if array else 0
    return [array[i * page_size : i * page_size + page_size] for i in range(n)]

def calcular_ventana_asistencia() -> tuple:
    """Retorna (fInicial, fFinal) como strings YYYYMMDDHHMMSS.
    Ajusta al día laboral anterior según día de la semana (array idéntico al JS original)."""
    DIAS_A_RESTAR = [2, 3, 1, 1, 1, 1, 1]  # JS getDay(): 0=Dom, 1=Lun, ...
    hoy = datetime.now()
    js_day = (hoy.weekday() + 1) % 7       # Python: Lun=0 → JS: Dom=0
    dia_anterior = hoy - timedelta(days=DIAS_A_RESTAR[js_day])
    return (
        dia_anterior.strftime('%Y%m%d') + '000000',
        hoy.strftime('%Y%m%d') + '235959'
    )

F_INICIAL, F_FINAL = calcular_ventana_asistencia()
logging.info(f'Ventana asistencia: {F_INICIAL} - {F_FINAL}')

# Validación
assert paginar([1,2,3], 2) == [[1,2],[3]], 'paginar KO'
assert paginar([], 100) == [], 'paginar vacío KO'
print(f'Ventana: {F_INICIAL} → {F_FINAL}')

In [ ]:
# Cell 8 — ObtenerAsistencia → LISTA_ASISTENCIA
#
# Fix #8 : extraer_hora_punch usa date[8:12] (no el quirk JS substring invertido)
# Fix #5 : safe_len() para Delay, BreakDelay, EarlyLeave, AccomplishedExtraTimeAfter
# Fix #1 : format_fecha('') retorna '' (no objeto datetime)
# Fix #2 : variable 'lista_usuarios' consistente, sin conflicto de nombres

async def obtener_asistencia(token: str, lista_usuarios: list) -> list:
    lista_completa = []
    paginas = paginar(lista_usuarios)

    for pagina in paginas:
        rut_str = ','.join(pagina)
        if not rut_str or not token:
            continue
        try:
            url = 'https://customerapi.geovictoria.com/api/v1/attendancebook'
            resp = requests.post(
                url,
                json={'StartDate': F_INICIAL, 'EndDate': F_FINAL, 'UserIds': rut_str},
                headers={
                    'Accept': 'application/json',
                    'Content-Type': 'application/json',
                    'Authorization': token
                },
                verify=False, timeout=120
            )
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            logging.error(f'ObtenerAsistencia - Error en servicio Asistencia: {e}')
            continue

        users = data.get('Users', [])
        if not users:
            logging.error('ObtenerAsistencia - No hay datos de Asistencia')
            continue

        for user in users:
            identifier = user.get('Identifier', '')
            for interval in user.get('PlannedInterval', []):
                punches = interval.get('Punches', [])
                shifts  = interval.get('Shifts', [])
                timeoffs = interval.get('TimeOffs', [])
                delay = interval.get('Delay', '')

                def get_hora(idx: int, tipo: str) -> str:
                    """Extrae hora de Punches[idx].Date si existe y el tipo coincide."""
                    if len(punches) > idx and punches[idx].get('Type') == tipo:
                        return extraer_hora_punch(punches[idx].get('Date', ''))
                    return ''

                def get_tipo(idx: int) -> str:
                    return punches[idx].get('Type', '') if len(punches) > idx else ''

                delay_val   = delay if safe_len(delay) > 0 else ''
                break_delay = interval.get('BreakDelay', '') if safe_len(delay) > 0 else ''
                early_leave = interval.get('EarlyLeave', '') if safe_len(delay) > 0 else ''
                hea_raw     = interval.get('AccomplishedExtraTimeAfter', '')
                hea_val     = hea_raw if safe_len(hea_raw) > 0 else ''

                fila = [
                    user.get('LastName', ''),                                          # Apellidos
                    user.get('Name', ''),                                              # Nombres
                    format_rut(identifier),                                            # Rut
                    user.get('GroupDescription', ''),                                  # Grupo
                    format_fecha(interval.get('Date', '')),                            # Fecha
                    timeoffs[0].get('TimeOffTypeDescription', '') if timeoffs else '',  # Permiso
                    shifts[0].get('ShiftDisplay', '') if shifts else '',               # Turno
                    get_hora(0, 'Ingreso'),                                            # Entro
                    get_tipo(0),                                                       # tipo
                    delay_val,                                                         # atraso
                    '',                                                                # Justificado
                    get_hora(1, 'Salida'),                                             # Salio
                    get_tipo(1),                                                       # tipo
                    get_hora(2, 'Ingreso'),                                            # Entro2
                    get_tipo(2),                                                       # tipo
                    break_delay,                                                       # atraso2
                    '',                                                                # Justificado2
                    get_hora(3, 'Salida'),                                             # Salio2
                    get_tipo(3),                                                       # tipo
                    early_leave,                                                       # adelanto
                    '',                                                                # Justificado3
                    hea_val,                                                           # HEA
                    '',                                                                # HEC
                    reemplazar(format_rut(identifier)),                                # HNT
                    identifier,                                                        # Identifier
                    user.get('PositionDescription', ''),                               # cargo
                ]
                lista_completa.append(fila)

    logging.info(f'ObtenerAsistencia - Total registros obtenidos: {len(lista_completa)}')
    return lista_completa

LISTA_ASISTENCIA = asyncio.run(obtener_asistencia(TOKEN, LISTA_USUARIOS))
logging.info(f'ObtenerAsistencia - Total: {len(LISTA_ASISTENCIA)}')

In [ ]:
# Cell 9 — ObtenerPermisosTipos → LISTA_PERMISOS_TIPOS
#
# Fix #10: variable 'req' muerta eliminada

async def obtener_permisos_tipos(token: str) -> list:
    url = 'https://customerapi.geovictoria.com/api/v1/TimeOff/GetTypes'
    try:
        resp = requests.post(
            url,
            headers={
                'Accept': 'application/json',
                'Content-Type': 'application/json',
                'Authorization': token
            },
            verify=False, timeout=30
        )
        resp.raise_for_status()
        data = resp.json()
        logging.info(f'ObtenerPermisos - Permisos Obtenidos: {len(data)}')
        lista = [
            [item['Id'], item.get('ExternalId', ''), item.get('TranslatedDescription', '')]
            for item in data
            if item.get('Status') == 'enabled'
        ]
        logging.info(f'ObtenerPermisos - Permisos Vigentes: {len(lista)}')
        return lista
    except Exception as e:
        logging.error(f'ObtenerPermisos - Error en servicio Obtener Permisos: {e}')
        return []

LISTA_PERMISOS_TIPOS = asyncio.run(obtener_permisos_tipos(TOKEN))

# Diccionario de búsqueda: nombre normalizado → TypeId
PERMISOS_DICT = {
    normalize_str(item[2]): item[0]
    for item in LISTA_PERMISOS_TIPOS
}

logging.info(f'ObtenerPermisos - Dict construido con {len(PERMISOS_DICT)} tipos')

In [ ]:
# Cell 10 — DescargarLicencias (Playwright: lmempleador.cl) → LICENCIAS_PDFS

async def descargar_licencias(carpetas: dict) -> list:
    hoy      = datetime.now()
    hace_7   = hoy - timedelta(days=7)
    f_desde  = hace_7.strftime('%d/%m/%Y')
    f_hasta  = hoy.strftime('%d/%m/%Y')
    downloads_dir = carpetas['downloads']
    pdfs = []

    logging.info('DescargaLicencias - Abrir Portal Licencias')
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=False)
        context = await browser.new_context(accept_downloads=True)
        page    = await context.new_page()
        try:
            await page.goto('https://www.lmempleador.cl/user/login/step2', wait_until='networkidle')
            await page.fill('input[type="email"]', os.environ['LM_EMAIL'])
            await page.fill('input[type="password"]', os.environ['LM_PASSWORD'])
            await page.click('button[type="submit"]')
            await page.wait_for_load_state('networkidle')

            for url_seccion in [
                'https://www.lmempleador.cl/licences',
                'https://www.lmempleador.cl/licences/2'
            ]:
                await page.goto(url_seccion, wait_until='networkidle')

                # Ingresar fechas
                for sel, val in [
                    ('input[name*="desde"], input[placeholder*="desde"], input[placeholder*="Desde"]', f_desde),
                    ('input[name*="hasta"], input[placeholder*="hasta"], input[placeholder*="Hasta"]', f_hasta),
                ]:
                    try:
                        await page.fill(sel, val)
                    except Exception:
                        pass

                await page.click('button:has-text("Filtrar")')
                await page.wait_for_load_state('networkidle')

                rows = await page.query_selector_all('table tbody tr')
                for row in rows:
                    btn = await row.query_selector('button:has-text("PDF")')
                    if not btn:
                        continue
                    try:
                        async with context.expect_download() as dl_info:
                            await btn.click()
                        dl   = await dl_info.value
                        dest = downloads_dir / dl.suggested_filename
                        await dl.save_as(dest)
                        pdfs.append(dest)
                        logging.info(f'DescargaLicencias - PDF descargado: {dl.suggested_filename}')
                    except Exception as e:
                        logging.error(f'DescargaLicencias - Error descargando PDF: {e}')

            # Cerrar sesión
            try:
                await page.click('a:has-text("Salir")')
            except Exception:
                pass

        except Exception as e:
            logging.error(f'DescargaLicencias - Error general: {e}')
        finally:
            await browser.close()

    logging.info(f'DescargaLicencias - Cerrar Portal Licencias')
    logging.info(f'DescargaLicencias - Total PDFs descargados: {len(pdfs)}')
    return pdfs

LICENCIAS_PDFS = asyncio.run(descargar_licencias(CARPETAS))

In [ ]:
# Cell 11 — ExtraerDatosPDF → LICENCIAS
#
# Fix #11: retorna lista acumulada explícitamente (JS nunca conectaba este resultado al Excel final)

def extraer_datos_pdf(pdf_files: list) -> list:
    # Buscar TypeId de licencia médica
    licencia_type_id = next(
        (item[0] for item in LISTA_PERMISOS_TIPOS if 'licencia' in item[2].lower()),
        ''
    )

    resultados = []
    for pdf_path in pdf_files:
        try:
            with pdfplumber.open(pdf_path) as pdf:
                texto = '\n'.join(p.extract_text() or '' for p in pdf.pages)

            run_m    = re.search(r'RUN[:\s]+(\d{1,2}\.\d{3}\.\d{3}-[\dkK])', texto, re.IGNORECASE)
            fecha_m  = re.search(r'FECHA INICIO(?:\s+DE)?\s+REPOSO[:\s]+(\d{2}/\d{2}/\d{4})', texto, re.IGNORECASE)
            dias_m   = re.search(r'N[°O]?\s*DE\s*D[IÍ]AS[:\s]+(\d+)', texto, re.IGNORECASE)

            if not (run_m and fecha_m):
                logging.warning(f'DescargaLicencias - No se encontraron datos en: {pdf_path.name}')
                continue

            rut       = rut_numerico(run_m.group(1))
            fecha_ini = datetime.strptime(fecha_m.group(1), '%d/%m/%Y')
            start_str = fecha_ini.strftime('%Y%m%d') + '000000'

            es_invalido = 'DOCUMENTO NO' in texto.upper() and 'V' in texto.upper() and 'LIDO' in texto.upper()

            if es_invalido or not dias_m:
                n_dias  = 1
                end_dt  = fecha_ini
            else:
                n_dias  = int(dias_m.group(1))
                end_dt  = fecha_ini + timedelta(days=n_dias - 1)

            end_str = end_dt.strftime('%Y%m%d') + '235959'
            resultados.append([rut, licencia_type_id, start_str, n_dias, end_str])
            logging.info(f'DescargaLicencias - RUT: {rut} Inicio: {start_str} Dias: {n_dias}')

            # Eliminar PDF después de procesar
            Path(pdf_path).unlink(missing_ok=True)
            logging.info(f'DescargaLicencias - Eliminar archivo descargado {pdf_path.name}')

        except Exception as e:
            logging.error(f'DescargaLicencias - Error procesando {pdf_path}: {e}')

    logging.info(f'DescargaLicencias - Licencias validadas: {len(resultados)}')
    return resultados

LICENCIAS = extraer_datos_pdf(LICENCIAS_PDFS)

In [ ]:
# Cell 12 — DescargarBUK (Playwright: maver.buk.cl) — condicional día > 5

async def descargar_buk(username: str) -> None:
    hoy     = datetime.now()
    ayer    = hoy - timedelta(days=1)
    f_desde = f'{ayer.day}-{ayer.month}-{ayer.year}'
    f_hasta = f'{hoy.day}-{hoy.month}-{hoy.year}'
    downloads = Path(f'C:/Users/{username}/Downloads')

    logging.info('Ws_Buk - Ingreso BUK')
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=False)
        context = await browser.new_context(accept_downloads=True)
        page    = await context.new_page()
        try:
            await page.goto('https://maver.buk.cl/users/sign_in', wait_until='networkidle')
            await page.fill('input[type="email"]', os.environ['BUK_EMAIL'])
            await page.click('button:has-text("Siguiente")')
            await page.fill('input[type="password"]', os.environ['BUK_PASSWORD'])
            await page.click('button:has-text("Iniciar Sesion")')
            await page.wait_for_load_state('networkidle')

            await page.goto('https://maver.buk.cl/report/custom_report_templates', wait_until='networkidle')

            # Reporte Vacaciones
            logging.info('Ws_Buk - Descargar excel Vacaciones')
            await page.click('text=Reporte Vacaciones Bot')
            await page.click('button:has-text("Descargar")')
            await page.wait_for_selector('input[placeholder*="Desde"], input[placeholder*="desde"]')
            await page.fill('input[placeholder*="Desde"], input[placeholder*="desde"]', f_desde)
            await page.fill('input[placeholder*="Hasta"], input[placeholder*="hasta"]', f_hasta)
            async with context.expect_download() as dl_info:
                await page.click('button:has-text("Descargar")')
            dl = await dl_info.value
            dest = downloads / dl.suggested_filename
            await dl.save_as(dest)
            logging.info(f'Ws_Buk - Archivo de Vacaciones: {dest.name}')
            await page.keyboard.press('Escape')

            # Reporte Permisos
            logging.info('Ws_Buk - Descargar excel Ausencias')
            await page.click('text=Reporte Permisos Bot')
            await page.click('button:has-text("Descargar")')
            async with context.expect_download() as dl_info2:
                await page.click('button:has-text("Descargar")')
            dl2 = await dl_info2.value
            dest2 = downloads / dl2.suggested_filename
            await dl2.save_as(dest2)
            logging.info(f'Ws_Buk - Archivo Ausencia: {dest2.name}')

            # Cerrar sesión
            logging.info('Ws_Buk - Cerrar Web')
            try:
                await page.click('text=Bot Maver')
                await page.click('text=Cerrar Sesion')
            except Exception:
                pass

        except Exception as e:
            logging.error(f'Ws_Buk - Error: {e}')
        finally:
            await browser.close()

if datetime.now().day > 5:
    asyncio.run(descargar_buk(USERNAME))
else:
    logging.info('Ws_Buk - Dia <= 5, no se descarga BUK')

In [ ]:
# Cell 13 — ProcesarVacaciones → VACACIONES
#
# Fix #4 (crítico): calcula EndDate = StartDate + N_dias (no devuelve raw 'Tipo' BUK en param[4])
# Fix #12: retorna [] si no hay archivo (no crash en concat posterior)

def procesar_vacaciones(username: str) -> list:
    downloads = Path(f'C:/Users/{username}/Downloads')
    excels = sorted(
        [f for f in downloads.iterdir()
         if 'reporte vacaciones bot' in f.name.lower() and f.suffix == '.xlsx'],
        key=lambda f: f.stat().st_mtime
    )
    if not excels:
        logging.warning('ObtenerListaVacaciones - No hay archivo de Vacaciones para procesar')
        return []

    excel_path = excels[-1]
    dest = PROJECT_ROOT / 'BUK/Buk vacaciones/BUK_vacaciones.xlsx'
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(excel_path, dest)
    logging.info(f'ObtenerListaVacaciones - Copiar {excel_path.name} a {dest}')

    vacaciones_type_id = next(
        (item[0] for item in LISTA_PERMISOS_TIPOS if 'vacaciones' in item[2].lower()),
        ''
    )

    resultados = []
    wb = openpyxl.load_workbook(dest, read_only=True, data_only=True)
    ws = wb.active

    for i, row in enumerate(ws.iter_rows(values_only=True), start=1):
        if i < 7:
            continue
        if not row[0]:
            break
        rut_raw, fecha_inicio, n_dias, estado, _ = row[0], row[1], row[2], row[3], row[4]

        if estado and 'rechazado' in str(estado).lower():
            continue

        rut_num = rut_numerico(str(rut_raw))

        start_dt = fecha_inicio if isinstance(fecha_inicio, datetime) \
                   else datetime.strptime(str(fecha_inicio)[:10], '%Y-%m-%d')
        n = int(n_dias) if n_dias else 1
        start_str = start_dt.strftime('%Y%m%d') + '000000'
        end_dt    = start_dt + timedelta(days=n)   # EndDate = StartDate + N días
        end_str   = end_dt.strftime('%Y%m%d') + '235959'

        resultados.append([rut_num, vacaciones_type_id, start_str, n, end_str])

    wb.close()
    logging.info(f'ObtenerListaVacaciones - Total a procesar: {len(resultados)}')
    return resultados

VACACIONES = procesar_vacaciones(USERNAME) if datetime.now().day > 5 else []

In [ ]:
# Cell 14 — ProcesarPermisosBUK → PERMISOS_BUK
#
# Fix #4 idem: calcula EndDate correctamente
# Nota: columnas BUK permisos: A=Rut, B=Dias, C=Fecha, D=Tipo (distinto orden que vacaciones)

def procesar_permisos_buk(username: str) -> list:
    downloads = Path(f'C:/Users/{username}/Downloads')
    excels = sorted(
        [f for f in downloads.iterdir()
         if 'reporte permisos bot' in f.name.lower() and f.suffix == '.xlsx'],
        key=lambda f: f.stat().st_mtime
    )
    if not excels:
        logging.warning('ObtenerListaAusencia - No hay archivo para procesar')
        return []

    excel_path = excels[-1]
    dest = PROJECT_ROOT / 'BUK/Permisos/BUK_permisos.xlsx'
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(excel_path, dest)
    logging.info(f'ObtenerListaAusencia - Copiar {excel_path.name} a {dest}')

    resultados = []
    wb = openpyxl.load_workbook(dest, read_only=True, data_only=True)
    ws = wb.active

    for i, row in enumerate(ws.iter_rows(values_only=True), start=1):
        if i < 7:
            continue
        if not row[0]:
            break
        rut_raw, n_dias, fecha_inicio, tipo_codigo = row[0], row[1], row[2], row[3]

        rut_num = rut_numerico(str(rut_raw))

        # Buscar TypeId en LISTA_PERMISOS_TIPOS por nombre normalizado
        tipo_norm = normalize_str(str(tipo_codigo))
        type_id = next(
            (item[0] for item in LISTA_PERMISOS_TIPOS if normalize_str(item[2]) == tipo_norm),
            None
        )
        if not type_id:
            logging.warning(f'ObtenerListaAusencia - Tipo no encontrado en GeoVictoria: {tipo_codigo}')
            continue

        start_dt = fecha_inicio if isinstance(fecha_inicio, datetime) \
                   else datetime.strptime(str(fecha_inicio)[:10], '%Y-%m-%d')
        n = int(n_dias) if n_dias else 1
        start_str = start_dt.strftime('%Y%m%d') + '000000'
        end_dt    = start_dt + timedelta(days=n - 1)  # 1 día = mismo día
        end_str   = end_dt.strftime('%Y%m%d') + '235959'

        resultados.append([rut_num, type_id, start_str, n, end_str])

    wb.close()
    logging.info(f'ObtenerListaAusencia - Total Ausencia a procesar: {len(resultados)}')
    return resultados

PERMISOS_BUK = procesar_permisos_buk(USERNAME) if datetime.now().day > 5 else []

In [ ]:
# Cell 15 — SubirPermisos
#
# Fix #4: param[4] ahora ES EndDate correcto (calculado en cells 11, 13, 14)
# Fix #12: PARAM_TOTAL siempre es lista válida (VACACIONES/PERMISOS_BUK/LICENCIAS = [] si no aplican)

async def subir_permisos(token: str, param_list: list) -> list:
    rut_maver = os.environ['RUT_MAVER']
    url = 'https://customerapi.geovictoria.com/api/v1/TimeOff/Upsert'
    exitosos = []
    errores  = []

    logging.info(f'SubirPermiso - Total a subir: {len(param_list)}')

    for param in param_list:
        if len(param) < 5:
            logging.warning(f'SubirPermiso - Parametro incompleto: {param}')
            continue

        logging.info(f'SubirPermiso - Validar datos lista {",".join(str(x) for x in param)}')
        try:
            resp = requests.post(
                url,
                json={
                    'UserIdentifier':        str(param[0]),
                    'TimeOffTypeId':         str(param[1]),
                    'StartDate':             str(param[2]),
                    'EndDate':               str(param[4]),  # param[4] = EndDate calculado
                    'CreatedByIdentifier':   rut_maver,
                    'Description':           '',
                    'StartTime':             '00:00',
                    'EndTime':               '23:59',
                    'Origin':                'API'
                },
                headers={
                    'Accept': 'application/json',
                    'Content-Type': 'application/json',
                    'Authorization': token
                },
                verify=False, timeout=30
            )
            resp.raise_for_status()
            result = resp.json()
            exitosos.append([param[0], param[1], result])
            logging.info(f'SubirPermiso - OK: {param[0]},{param[1]}')
        except Exception as e:
            logging.error(f'SubirPermiso - Error en servicio Subir Permiso: {e} Rut: {param[0]}')
            errores.append(param)

    logging.info(f'SubirPermiso - Total subidos: {len(exitosos)}')
    logging.info(f'SubirPermiso - Total errores: {len(errores)}')
    return exitosos

# Fix #12: todas las listas están inicializadas a [] si no aplican
PARAM_TOTAL = VACACIONES + PERMISOS_BUK + LICENCIAS

if PARAM_TOTAL:
    asyncio.run(subir_permisos(TOKEN, PARAM_TOTAL))
else:
    logging.info('SubirPermiso - No hay informacion del permiso')

In [ ]:
# Cell 16 — ReporteFinal → RET
#
# Fix #6: re.sub(r'-', '.', ...) reemplaza TODOS los guiones (JS .replace('-','.') solo el primero)

def reporte_final(data: list) -> list:
    hoy = datetime.now()
    DIAS_A_RESTAR = [2, 3, 1, 1, 1, 1, 1]
    js_day = (hoy.weekday() + 1) % 7
    dia_anterior = hoy - timedelta(days=DIAS_A_RESTAR[js_day])

    hoy_str          = hoy.strftime('%d-%m-%Y')
    dia_anterior_str = dia_anterior.strftime('%d-%m-%Y')

    COLS = {'FECHA': 4, 'PERMISO': 5, 'TURNO': 6, 'ENTRO': 7}

    def set_presente_ausente(fila: list) -> list:
        if fila[COLS['PERMISO']] == '':
            fila[COLS['PERMISO']] = 'Presente' if fila[COLS['ENTRO']] else 'Ausencia'
        return fila

    def es_turno_manana(s: str) -> bool:
        return any(str(s).startswith(p) for p in ('06', '07', '08', '09'))

    def es_turno_tarde(s: str) -> bool:
        return any(str(s).startswith(p) for p in ('16', '17', '20', '22'))

    def filtro_turnos(fila: list) -> bool:
        fecha = str(fila[COLS['FECHA']])
        turno = str(fila[COLS['TURNO']])
        if hoy_str in fecha and es_turno_manana(turno):
            return True
        if dia_anterior_str in fecha and es_turno_tarde(turno):
            return True
        return False

    def fec(dato: list) -> list:
        """Fix #6: re.sub global reemplaza todos los guiones, no solo el primero."""
        dato[4] = re.sub(r'-', '.', str(dato[4]))
        return dato

    logging.info(f'ReporteFinal_Ws - Total de datos a procesar: {len(data)}')

    ret = [set_presente_ausente(list(fila)) for fila in data]
    ret = [fila for fila in ret if filtro_turnos(fila)]
    ret = [fec(fila) for fila in ret]

    logging.info(f'ReporteFinal_Ws - Total de registros a informar: {len(ret)}')
    return ret

RET = reporte_final(LISTA_ASISTENCIA)
print(f'Registros para el Excel: {len(RET)}')

In [ ]:
# Cell 17 — DescargarPermisosGeo (90 días) → PERMISOS_GEO_RAW
#
# Fix #2/#3: variable única 'lista_usuarios', paginar() retorna directamente

async def descargar_permisos_geo(token: str, lista_usuarios: list, username: str) -> list:
    hoy     = datetime.now()
    hace_90 = hoy - timedelta(days=90)
    f_ini   = hace_90.strftime('%Y%m%d') + '000000'
    f_fin   = hoy.strftime('%Y%m%d') + '235959'

    # Cargar filtro desde GeoVictoria/Permisos.json en la raíz del proyecto
    permisos_json = PROJECT_ROOT / 'GeoVictoria/Permisos.json'
    filtros_norm = []
    if permisos_json.exists():
        with open(permisos_json, 'r', encoding='utf-8') as f:
            filtros_norm = [normalize_str(x) for x in json.load(f)]

    lista_total = []
    paginas = paginar(lista_usuarios)  # Fix #3: paginar() devuelve lista, no variable undefined

    for pagina in paginas:
        rut_str = ','.join(pagina)
        if not rut_str or not token:
            continue
        try:
            url = 'https://customerapi.geovictoria.com/api/v1/TimeOff/Get'
            resp = requests.post(
                url,
                json={'StartDate': f_ini, 'EndDate': f_fin, 'UserIds': rut_str},
                headers={
                    'Accept': 'application/json',
                    'Content-Type': 'application/json',
                    'Authorization': token
                },
                verify=False, timeout=120
            )
            resp.raise_for_status()
            data = resp.json()

            for item in (data if isinstance(data, list) else []):
                desc = normalize_str(item.get('TimeOffTypeDescription', ''))
                ends = item.get('Ends', '')
                if (not filtros_norm or desc in filtros_norm) and ends >= f_fin:
                    lista_total.append(item)
        except Exception as e:
            logging.error(f'DescargaPermisos - Error en servicio Permisos: {e}')

    logging.info(f'DescargaPermisos - Total permisos obtenidos: {len(lista_total)}')
    return lista_total

PERMISOS_GEO_RAW = asyncio.run(descargar_permisos_geo(TOKEN, LISTA_USUARIOS, USERNAME))

In [ ]:
# Cell 18 — ObtenerUsuariosDetalle + GenerarListaPermisosFinal → LISTA_PERMISOS_FINAL

async def obtener_usuarios_detalle(token: str) -> list:
    """Segunda llamada a User/List retornando objetos completos (nombre, apellidos, cargo)."""
    url = 'https://apiv3.geovictoria.com/api/User/List'
    try:
        resp = requests.post(
            url,
            headers={
                'Accept': 'application/json',
                'Content-Type': 'application/json',
                'Authorization': OAUTH_HEADER
            },
            timeout=30
        )
        resp.raise_for_status()
        data = resp.json()
        return [
            {
                'id':       item['Identifier'],
                'nombre':   item.get('Name', ''),
                'apellidos': item.get('LastName', ''),
                'cargo':    item.get('positionName', '')
            }
            for item in data
            if item.get('Enabled') == 1
            and 'casino' not in (item.get('GroupDescription') or '').lower()
        ]
    except Exception as e:
        logging.error(f'ObtenerUsuarios - Error en servicio Usuarios en Descarga Permisos: {e}')
        return []

def generar_lista_permisos_final(usuarios_detalle: list, lista_permisos: list) -> list:
    def fec_perm(f: str) -> str:
        s = str(f)
        return s[6:8] + '-' + s[4:6] + '-' + s[0:4] if len(s) >= 8 else ''

    def dif(ini: str, fin: str) -> int:
        try:
            i = datetime.strptime(str(ini)[:8], '%Y%m%d')
            f = datetime.strptime(str(fin)[:8], '%Y%m%d')
            return abs((f - i).days)
        except Exception:
            return 0

    usuarios_idx = {u['id']: u for u in usuarios_detalle}
    resultado = []

    for perm in lista_permisos:
        uid = perm.get('UserIdentifier', '')
        user = usuarios_idx.get(uid)
        if not user:
            continue
        resultado.append([
            user['apellidos'],
            user['nombre'],
            format_rut(user['id']),
            user['cargo'],
            perm.get('TimeOffTypeDescription', ''),
            '',
            fec_perm(perm.get('Starts', '')),
            fec_perm(perm.get('Ends', '')),
            '', '', '',
            dif(perm.get('Starts', ''), perm.get('Ends', ''))
        ])

    logging.info(f'Total Permisos Descargados: {len(resultado)}')
    return resultado

USUARIOS_DETALLE  = asyncio.run(obtener_usuarios_detalle(TOKEN))
LISTA_PERMISOS_FINAL = generar_lista_permisos_final(USUARIOS_DETALLE, PERMISOS_GEO_RAW)
print(f'Permisos para Excel: {len(LISTA_PERMISOS_FINAL)}')

In [ ]:
# Cell 19 — EscribirExcel → Informe de Asistencia VersionFinal.xlsx
#
# xlwings (COM) en lugar de openpyxl porque openpyxl 3.1.5 falla en este Excel
# con tablas dinámicas (bug Nested.from_tree()).

def escribir_excel(ret: list, lista_permisos_final: list) -> None:
    excel_path = r'c:\dev\Asistencia\Informe de Asistencia VersionFinal.xlsx'

    app = xw.App(visible=False)
    try:
        wb = app.books.open(excel_path)

        # --- Hoja: Base datos ---
        ws_base = wb.sheets['Base datos']
        ultima_col = max(len(r) for r in ret) if ret else 1
        ws_base.range('A3').expand('table').clear_contents()
        if ret:
            ws_base.range('A3').value = ret
            logging.info(f'EscribirExcel - Base datos: {len(ret)} filas escritas')

        # --- Hoja: Detalle Licencias Médicas ---
        ws_lic = None
        for nombre_hoja in ['Detalle Licencias Médicas', 'Detalle Licencias M\xe9dicas',
                             'Detalle Licencias Medicas']:
            try:
                ws_lic = wb.sheets[nombre_hoja]
                break
            except Exception:
                continue

        if ws_lic is not None:
            ws_lic.range('A2').expand('table').clear_contents()
            if lista_permisos_final:
                ws_lic.range('A2').value = lista_permisos_final
            logging.info(f'EscribirExcel - Detalle Licencias: {len(lista_permisos_final)} filas escritas')
        else:
            logging.warning('EscribirExcel - Hoja Detalle Licencias Médicas no encontrada')

        # --- Hoja: Asistencia Maver — actualizar tablas dinámicas ---
        wb.api.RefreshAll()
        logging.info('EscribirExcel - Tablas dinamicas actualizadas')

        wb.save()
        wb.close()
        logging.info(f'EscribirExcel - Guardado: {excel_path}')

    except Exception as e:
        logging.error(f'EscribirExcel - Error: {e}')
        raise
    finally:
        app.quit()

escribir_excel(RET, LISTA_PERMISOS_FINAL)
logging.info('Bot Asistencia - Fin Ejecucion')

In [ ]:
# Cell 20 — main() — Orquestación completa (ejecutar todo de una vez)

async def main():
    username = obtener_usuario()
    setup_log(username)
    logging.info('Bot Asistencia - Inicio Ejecucion')

    respaldar_downloads(username)

    token = await obtener_token()
    assert len(token) > 10, 'Token GeoVictoria inválido'

    lista_usuarios = await obtener_usuarios(token)
    carpetas       = generar_carpetas(username)

    lista_asistencia   = await obtener_asistencia(token, lista_usuarios)
    lista_permisos_tipos_local = await obtener_permisos_tipos(token)

    licencias_pdfs = await descargar_licencias(carpetas)
    licencias      = extraer_datos_pdf(licencias_pdfs)

    # Fix #12: vacaciones/permisos inicializados a [] antes del guard
    vacaciones:  list = []
    permisos_buk: list = []
    if datetime.now().day > 5:
        await descargar_buk(username)
        vacaciones   = procesar_vacaciones(username)
        permisos_buk = procesar_permisos_buk(username)

    param = vacaciones + permisos_buk + licencias
    if param:
        await subir_permisos(token, param)

    ret = reporte_final(lista_asistencia)

    permisos_geo_raw    = await descargar_permisos_geo(token, lista_usuarios, username)
    usuarios_detalle    = await obtener_usuarios_detalle(token)
    lista_permisos_final = generar_lista_permisos_final(usuarios_detalle, permisos_geo_raw)

    escribir_excel(ret, lista_permisos_final)
    logging.info('Bot Asistencia - Fin Ejecucion')

# Descomentar para ejecutar todo desde cero:
# asyncio.run(main())